# 03 — Solar Radiation, VPD, and PTQ Features

Derives stage-specific stress and energy features from the new Daymet variables (srad, vp, swe, vpd, ptq).

**New features added per field-year:**

| Feature | Window | Rationale |
|---|---|---|
| `srad_greenup_mean` | DOS 240-290 (Mar-Apr) | Light availability during stem elongation |
| `srad_grain_mean` | DOS 295-355 (May-Jun) | Light during grain filling |
| `vpd_max_post_anthesis` | DOS 320-360 (post-bloom) | Drought stress during grain fill |
| `vpd_mean_greenup` | DOS 240-290 | Spring drought stress |
| `ptq_grain_mean` | DOS 295-355 | Photothermal quotient (grain yield proxy) |
| `swe_max_winter` | DOS 150-220 (Dec-Jan) | Snow cover protection |

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Paths
FEAT_IN  = 'data/processed/features/features_with_aptt.parquet'
FEAT_OUT = 'data/processed/features/features_with_solar_vpd.parquet'
WX_PATH  = 'data/raw/satellite/daymet_daily_weather_full.csv'
MAP_PATH = 'data/processed/csb_to_buf_mapping.csv'

feat = pd.read_parquet(FEAT_IN)
wx = pd.read_csv(WX_PATH)
wx['date'] = pd.to_datetime(wx['date'], format='mixed')
wx['year'] = wx['date'].dt.year
wx['doy'] = wx['date'].dt.dayofyear
wx['FIELDID'] = wx['FIELDID'].astype(str)
if 'tmin' in wx.columns: wx = wx.rename(columns={'tmin':'Tmin','tmax':'Tmax'})
wx['T_mean'] = (wx['Tmin'] + wx['Tmax']) / 2

mapping = {}
if os.path.exists(MAP_PATH):
    m_df = pd.read_csv(MAP_PATH)
    mapping = dict(zip(m_df['field_id'].astype(str), m_df['nearest_BUF'].astype(str)))

print(f'Features: {feat.shape}, Weather: {wx.shape}')
print(f'New weather columns: {[c for c in wx.columns if c in ["srad","vp","swe","vpd","ptq"]]}')

## 1. Compute DOS (day of season) per weather row

Same convention as multi-stage validation: DOS=1 = July 1 of growing-season-start year.

In [ ]:
# Each weather row needs: DOS within the growing season(s) it belongs to
# Growing season N-1→N starts Jul 1 of year N-1, ends Jun 30 of year N
# A weather row in year N belongs to:
#   - growing_season N-1→N if month <= 6 (Jan-Jun → DOS 184+ from Jul 1 of N-1)
#   - growing_season N→N+1 if month >= 7 (Jul-Dec → DOS 1+ from Jul 1 of N)

wx['month'] = wx['date'].dt.month
wx['gs_start_year'] = np.where(wx['month'] >= 7, wx['year'], wx['year'] - 1)
wx['harvest_year']  = wx['gs_start_year'] + 1
wx['gs_start_date'] = pd.to_datetime(wx['gs_start_year'].astype(str) + '-07-01')
wx['dos'] = (wx['date'] - wx['gs_start_date']).dt.days + 1

print('DOS sample:')
print(wx[['date','year','harvest_year','dos']].head(5).to_string(index=False))
print(f'\nDOS range: {wx["dos"].min()} → {wx["dos"].max()}')

## 2. Compute window-aggregated features per (field, harvest_year)

In [ ]:
# Define DOS windows for each feature
WINDOWS = {
    'srad_greenup_mean':       {'var': 'srad', 'agg': 'mean', 'dos': (240, 290)},
    'srad_grain_mean':         {'var': 'srad', 'agg': 'mean', 'dos': (295, 355)},
    'vpd_max_post_anthesis':   {'var': 'vpd',  'agg': 'max',  'dos': (320, 360)},
    'vpd_mean_greenup':        {'var': 'vpd',  'agg': 'mean', 'dos': (240, 290)},
    'ptq_grain_mean':          {'var': 'ptq',  'agg': 'mean', 'dos': (295, 355)},
    'swe_max_winter':          {'var': 'swe',  'agg': 'max',  'dos': (150, 220)},
}

# Compute per-field-year features
results = []
all_field_years = feat[['field_id', 'year']].drop_duplicates().values
print(f'Computing {len(WINDOWS)} new features for {len(all_field_years)} field-years...')

for idx, (fid, yr) in enumerate(all_field_years):
    fid = str(fid); yr = int(yr)
    wx_fid = mapping.get(fid, fid)
    sub = wx[(wx['FIELDID']==wx_fid) & (wx['harvest_year']==yr)]
    if len(sub) == 0:
        continue
    row = {'field_id': fid, 'year': yr}
    for feat_name, spec in WINDOWS.items():
        win = sub[(sub['dos'] >= spec['dos'][0]) & (sub['dos'] <= spec['dos'][1])]
        if len(win) == 0:
            row[feat_name] = np.nan
        else:
            v = win[spec['var']].dropna()
            if len(v) == 0:
                row[feat_name] = np.nan
            elif spec['agg'] == 'mean':
                row[feat_name] = float(v.mean())
            elif spec['agg'] == 'max':
                row[feat_name] = float(v.max())
            elif spec['agg'] == 'sum':
                row[feat_name] = float(v.sum())
    results.append(row)
    if (idx+1) % 500 == 0:
        print(f'  {idx+1}/{len(all_field_years)} done')

new_feat = pd.DataFrame(results)
print(f'\nNew features computed: {new_feat.shape}')
print('\nSummary statistics:')
print(new_feat.describe().round(2))

## 3. Correlations with target

In [ ]:
# Quick check — do these features correlate with flag_true_doy?
merged = feat[['field_id','year','flag_true_doy']].merge(
    new_feat, on=['field_id','year'], how='inner')
print('Correlations with flag_true_doy:')
for col in WINDOWS:
    sub = merged[[col, 'flag_true_doy']].dropna()
    if len(sub) < 50:
        print(f'  {col:30s}  n={len(sub)}  insufficient data')
        continue
    r = np.corrcoef(sub[col], sub['flag_true_doy'])[0,1]
    print(f'  {col:30s}  n={len(sub):4d}  r={r:+.3f}')

## 4. Merge and save

In [ ]:
feat_full = feat.merge(new_feat, on=['field_id','year'], how='left')
feat_full.to_parquet(FEAT_OUT, index=False)
print(f'Saved: {FEAT_OUT}')
print(f'Shape: {feat_full.shape}')
print(f'New columns added: {list(WINDOWS.keys())}')
missing = feat_full[list(WINDOWS.keys())].isna().mean() * 100
print(f'\nMissing % per new feature:')
for c, pct in missing.items():
    print(f'  {c:30s}  {pct:.1f}% missing')